# Module 09 — The Learning Rate (notebook)

Walkthrough of [`schedule.py`](schedule.py). We'll:

1. Visualize the canonical schedules — cosine, WSD, and constant — at the same warmup/total.
2. Verify that the scheduler preserves per-group LR ratios (so muP works through warmup + decay).
3. Train a tiny model **with vs without** a scheduler and compare loss curves.
4. Reproduce the four loss-curve signatures from § 6 of the README — too-high LR, too-low LR, short warmup, aggressive decay — on a small synthetic task.

**Compute:** CPU is enough.  
**Time:** ~10 minutes.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Module 09 — schedulers (the only file this module owns).
from schedule import (
    WarmupCosineLR, WSDLR, ConstantWithWarmupLR,
    ScheduleConfig, build_scheduler,
)

torch.manual_seed(0)
print("torch:", torch.__version__)

## 1. Visualize the schedules

Replay each scheduler's `get_lr()` over its full trajectory. All three at the same hyperparameters except the schedule shape.

In [ ]:
def trace_schedule(sched_cls, n_steps, **kwargs):
    """Step a fresh scheduler `n_steps` times and return the LR trajectory."""
    p = torch.zeros(1, requires_grad=True)
    opt = torch.optim.SGD([p], lr=1.0)
    s = sched_cls(opt, **kwargs)
    lrs = [opt.param_groups[0]['lr']]
    for _ in range(n_steps - 1):
        opt.step(); s.step()
        lrs.append(opt.param_groups[0]['lr'])
    return lrs

T = 1000
WU = 100
MR = 0.1

cosine = trace_schedule(WarmupCosineLR, T,
                        warmup_steps=WU, total_steps=T, min_lr_ratio=MR)
wsd    = trace_schedule(WSDLR,          T,
                        warmup_steps=WU, total_steps=T, decay_steps=150, min_lr_ratio=MR)
const  = trace_schedule(ConstantWithWarmupLR, T, warmup_steps=WU)

plt.figure(figsize=(9, 4))
plt.plot(cosine, label="WarmupCosine (peak->min cosine)", lw=2)
plt.plot(wsd,    label="WSD (warmup-stable-decay)",       lw=2)
plt.plot(const,  label="ConstantWithWarmup",              lw=2, alpha=0.7)
plt.axvline(WU, color="gray", linestyle=":", alpha=0.5, label=f"warmup_steps = {WU}")
plt.axvline(T-150, color="red", linestyle=":", alpha=0.3, label="WSD decay start")
plt.xlabel("step"); plt.ylabel("LR (peak = 1.0)")
plt.title("LR schedules at total_steps=1000, warmup=100, min_lr_ratio=0.1")
plt.legend(fontsize=9, loc="lower left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Read the shapes:

- **Cosine** spends a long fraction of training at LR > 80% of peak, then a smooth tail to 10%. Optimal when `total_steps` is fixed.
- **WSD** holds at peak for the bulk of training, then a short fast decay. Lets you extend training without re-shaping the cosine. DeepSeek-V3's choice.
- **Constant** is the ablation baseline. Don't use it for real runs.

## 2. The scheduler preserves per-group LR ratios (muP-safe)

muP gives different param groups different *initial* LRs. The scheduler multiplies each group's initial_lr by the same time-varying factor — so the ratio between groups is preserved at every step. Verify this on a 3-group setup that mimics what muP would do.

In [ ]:
# Three groups with different initial LRs (think: embedding, hidden, output under muP).
p_emb = torch.zeros(8, requires_grad=True)
p_hid = torch.zeros(8, requires_grad=True)
p_out = torch.zeros(8, requires_grad=True)

opt = torch.optim.SGD([
    {"params": [p_emb], "lr": 3e-3, "name": "embedding"},
    {"params": [p_hid], "lr": 1e-3, "name": "hidden"},    # ratio 1/3 to embedding
    {"params": [p_out], "lr": 3e-3, "name": "output"},
])
sched = WarmupCosineLR(opt, warmup_steps=20, total_steps=200, min_lr_ratio=0.1)

trajectories = {g["name"]: [] for g in opt.param_groups}
for step in range(200):
    for g in opt.param_groups:
        trajectories[g["name"]].append(g["lr"])
    opt.step(); sched.step()

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name, lrs in trajectories.items():
    ax[0].plot(lrs, label=name, lw=2)
ax[0].set_xlabel("step"); ax[0].set_ylabel("LR")
ax[0].set_title("Per-group LRs through the schedule")
ax[0].legend(); ax[0].grid(alpha=0.3)

# Ratio of hidden/embedding LR — should stay at exactly 1/3.
# Skip step 0 (LR=0 at start of warmup gives 0/0).
ratio = [h / e for h, e in zip(trajectories["hidden"][1:], trajectories["embedding"][1:])]
ax[1].plot(range(1, 200), ratio, color="C3", lw=2)
ax[1].axhline(1/3, color="black", linestyle=":", label="target ratio = 1/3")
ax[1].set_xlabel("step"); ax[1].set_ylabel("hidden_lr / embedding_lr")
ax[1].set_title("muP per-group ratio is invariant through schedule")
ax[1].set_ylim(0.0, 0.5); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Sanity: ratio shouldn't drift.
print(f"hidden/embedding ratio range: [{min(ratio):.6f}, {max(ratio):.6f}]  (target 0.3333)")

The ratio is exactly $1/3$ at every step. This is why muP and standard LR schedulers compose without any special integration — the cosine just multiplies all groups by the same factor.

## 3. Tiny training run — with vs without a scheduler

Train two identical small models for the same number of steps. One uses WarmupCosineLR; the other uses constant LR (no decay). Compare final losses.

*The synthetic dataset doesn't let us measure validation loss meaningfully, so this is mostly about seeing the scheduled LR's effect on training-loss shape.*

In [ ]:
# A tiny, self-contained transformer for these demos. ~30k params. Module 09
# focuses on the scheduler — we don't import a model from elsewhere, we
# just define a small one here.
class TinyLM(nn.Module):
    def __init__(self, vocab=256, d=64, n_heads=4, max_seq=32):
        super().__init__()
        self.tok = nn.Embedding(vocab, d)
        self.pos = nn.Embedding(max_seq, d)
        self.norm1 = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, bias=False, batch_first=True)
        self.norm2 = nn.LayerNorm(d)
        self.ffn = nn.Sequential(
            nn.Linear(d, 4*d, bias=False), nn.GELU(), nn.Linear(4*d, d, bias=False),
        )
        self.norm_f = nn.LayerNorm(d)
        # Tied LM head.
        self.lm_head = nn.Linear(d, vocab, bias=False)
        self.lm_head.weight = self.tok.weight
        # Standard init (std=0.02 throughout).
        nn.init.normal_(self.tok.weight, std=0.02)
        nn.init.normal_(self.pos.weight, std=0.02)
        for m in [self.attn.in_proj_weight, self.attn.out_proj.weight,
                  self.ffn[0].weight, self.ffn[2].weight]:
            nn.init.normal_(m, std=0.02)

    def forward(self, ids):
        T = ids.size(1)
        pos = torch.arange(T, device=ids.device).unsqueeze(0)
        x = self.tok(ids) + self.pos(pos)
        n = self.norm1(x)
        a, _ = self.attn(n, n, n, is_causal=True, need_weights=False,
                         attn_mask=nn.Transformer.generate_square_subsequent_mask(T, device=ids.device))
        x = x + a
        x = x + self.ffn(self.norm2(x))
        return self.lm_head(self.norm_f(x))

def random_batch(B, T, vocab, gen=None):
    """Random next-token-prediction batch. Trivial task, enough to show LR effects."""
    ids = torch.randint(0, vocab, (B, T+1), generator=gen)
    return ids[:, :-1], ids[:, 1:]

def small_run(use_scheduler: bool, total_steps: int = 100, lr: float = 3e-3,
              vocab: int = 256, seq: int = 32, batch: int = 8):
    """Train a TinyLM with or without a cosine schedule. Returns (losses, lrs)."""
    torch.manual_seed(0)
    model = TinyLM(vocab=vocab, d=64, n_heads=4, max_seq=seq)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    if use_scheduler:
        scheduler = WarmupCosineLR(optimizer, warmup_steps=10,
                                   total_steps=total_steps, min_lr_ratio=0.1)
    gen = torch.Generator().manual_seed(42)
    losses, lrs = [], []
    for step in range(total_steps):
        ids, tgt = random_batch(batch, seq, vocab, gen)
        logits = model(ids)
        loss = F.cross_entropy(logits.reshape(-1, vocab), tgt.reshape(-1))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if use_scheduler:
            scheduler.step()
        losses.append(loss.item())
        lrs.append(optimizer.param_groups[0]["lr"])
    return losses, lrs

print("Training without scheduler (constant LR=3e-3)...")
loss_const, lr_const = small_run(use_scheduler=False)
print("Training with WarmupCosineLR (peak=3e-3, warmup=10, total=100, min=10%)...")
loss_sched, lr_sched = small_run(use_scheduler=True)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(loss_const, label="constant LR",     lw=2, alpha=0.7)
ax[0].plot(loss_sched, label="cosine schedule", lw=2)
ax[0].set_xlabel("step"); ax[0].set_ylabel("loss")
ax[0].set_title("Training loss"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(lr_const,  label="constant LR",     lw=2, alpha=0.7)
ax[1].plot(lr_sched,  label="cosine schedule", lw=2)
ax[1].set_xlabel("step"); ax[1].set_ylabel("LR")
ax[1].set_title("LR over training"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"\nFinal loss (constant LR): {loss_const[-1]:.4f}")
print(f"Final loss (cosine):       {loss_sched[-1]:.4f}")

On a 100-step toy task with random data, the difference between schedules is modest — the model hasn't really converged anyway. The real win from scheduling shows up over thousands of steps on real data, where:

- The warmup phase prevents early divergence.
- The constant-high-LR middle phase makes fast progress.
- The cosine tail lets the model settle into a sharper minimum at the end.

Module 11's run on FineWeb-Edu will produce the full-shape loss curve where each region's contribution is visible.

## 4. Loss-curve diagnosis — what bad LRs look like

Reproduce the four failure modes from § 6 of the README. We'll deliberately mis-set the LR to produce each signature, then plot them side by side.

The model is the same tiny one as above; only the LR / warmup / schedule changes.

In [ ]:
def run_loss_curve(lr, warmup, total=150, schedule_kind="cosine",
                    vocab=256, seq=32, batch=8):
    """Run a small training trace and return the loss curve. `schedule_kind`
    is 'cosine', 'constant', or 'none' (no schedule, raw LR). Uses the same
    TinyLM defined above."""
    torch.manual_seed(0)
    model = TinyLM(vocab=vocab, d=64, n_heads=4, max_seq=seq)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    if schedule_kind == "cosine":
        sched = WarmupCosineLR(optimizer, warmup_steps=warmup, total_steps=total, min_lr_ratio=0.1)
    elif schedule_kind == "constant":
        sched = ConstantWithWarmupLR(optimizer, warmup_steps=warmup)
    else:
        sched = None
    gen = torch.Generator().manual_seed(42)
    losses = []
    for _ in range(total):
        ids, tgt = random_batch(batch, seq, vocab, gen)
        logits = model(ids)
        loss = F.cross_entropy(logits.reshape(-1, vocab), tgt.reshape(-1))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if sched is not None:
            sched.step()
        losses.append(loss.item())
    return losses

# Run each failure-mode + a baseline (matching § 6 of the README).
loss_good       = run_loss_curve(lr=3e-3,  warmup=20, total=150, schedule_kind="cosine")
loss_too_high   = run_loss_curve(lr=3e-1,  warmup=20, total=150, schedule_kind="cosine")  # 100x too high
loss_too_low    = run_loss_curve(lr=1e-5,  warmup=20, total=150, schedule_kind="cosine")  # 300x too low
loss_short_warm = run_loss_curve(lr=3e-2,  warmup=1,  total=150, schedule_kind="cosine")  # high lr with no warmup

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
panels = [
    ("Healthy: LR=3e-3, warmup=20, cosine", loss_good,       "C0"),
    ("LR too HIGH (3e-1): oscillation",      loss_too_high,   "C3"),
    ("LR too LOW (1e-5): slow + smooth",     loss_too_low,    "C2"),
    ("Warmup too short (1 step, LR=3e-2)",   loss_short_warm, "C4"),
]
for ax, (title, losses, color) in zip(axes.flat, panels):
    ax.plot(losses, color=color, lw=1.5)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel("loss")
    ax.grid(alpha=0.3)
for ax in axes[1]:
    ax.set_xlabel("step")
plt.tight_layout(); plt.show()

Reading the four signatures:

- **Healthy** (top left): smooth monotone decrease, reaches a sensible loss.
- **Too HIGH** (top right): loss may spike, oscillate, or NaN. Sometimes the model recovers; sometimes it doesn't.
- **Too LOW** (bottom left): smooth decay — but to a clearly worse final loss than the healthy run. The diagnostic is that the curve is *too smooth* and the grad norm (not shown but it's there in `train_step`'s return) is tiny.
- **Warmup too short** (bottom right): an early bump or restart in the loss curve as the LR jumps to peak before AdamW's running statistics have stabilized.

**The single most useful skill in pretraining diagnostics is recognizing these four shapes.** When something looks weird, your first hypothesis is the LR; check the schedule and the grad norm before suspecting anything else.

## Recap

You now have:

- A working `WarmupCosineLR`, `WSDLR`, and `ConstantWithWarmupLR` in `schedule.py`.
- A `build_scheduler(optimizer, cfg)` factory that pairs with the integrator ([Module 11's `train.py`](../11-pretraining-in-practice/train.py)) in two added lines.
- Proof that the scheduler preserves muP's per-group LR ratios (so muP transfer composes cleanly with cosine/WSD).
- Empirical loss-curve signatures for the four common LR mistakes.

**Next:** [Module 10 — Scaling and Efficiency](../10-scaling-and-efficiency/). The FSDP2 sharding stages, mixed-precision policies, gradient checkpointing, Chinchilla compute laws, MTP — everything that makes the framework throughput-aware.